# Time Operators

Streaming data is infinite. Kafi Streams is an in-memory stream processor. Memory is never infinite.

So of course you need *time operators* that effectively clean up memory so that your Kafi Streams processing pipeline has constant, not ever-growing memory usage.

Freeing memory of timed out data is implemented as [expiry](#expiry) in Kafi Streams.

The time operators also allow you to implement the [*time windows*](#windows) as e.g. in Kafka Streams, such as [tumbling](#tumbling), [hopping](#hopping), [cumulative](#cumulative), [sliding](#sliding) and [session](#session) windows. And moreover, Kafi Streams enables you to build arbitrary [new types of time windows](#custom) as well.

## Overview

[Preparation](#prep)

* [Expiry](#expiry)
  * [expire()](#expire-operator)
* [Time windows](#windows)
  * [Time Windows = expire + group + trigger](#expire_group_trigger)
  * [Tumbling windows](#tumbling)
  * [Hopping windows](#hopping)
  * [Cumulative windows](#cumulative)
  * [Sliding windows](#sliding)
  * [Session windows](#session)
  * [Custom windows](#custom)
  

---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [11]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

click_source_str = "clicks"
customer_source_str = "customers"



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Please also note that when we re-use the same example over and over again to illustrate how the operators work, we always mark the important new parts as follows:
```python
    # <------------------------------>
    ...important new parts...
    # <------------------------------>
```

---
<a id="expiry"></a>
## Expiry

Expiry is the central concept in Kafi Streams for freeing memory of timed out data.

Thanks to pydbsp, Kafi Streams can implement expiry natively, without having to bolt on any kind of mechanism on top.

Essentially, expiry has to be defined only once for each source at the beginning of the Kafi Streams topology. All the stateful operators downstream do not need any special handling - they are automatically cleaned up by the expired records percolating through the topology, one by one.

In classical stream processing lingo, message expiry can be likened to a "sliding window" where each individual message times out on its own.

We need an example. Let us recollect the example from the [Quickstart](../quickstart.ipynb) using the `TopologyNode` class (see [Architecture](../architecture.ipynb))


In [ ]:
click_source_str = "clicks"
customer_source_str = "customers"

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

joined_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
)

built_tn = Tn.build(joined_tn)


And then, let us throw data at it and see how the global state size of the topology grows:

In [12]:
def run():    
    sink_m_list = []
    for i in range(100):
        # 1. Generate new data.
        click_m_list = click_generator.generate(100)
        customer_m_list = customer_generator.generate(100)

        # 2. Push the new data to the topology + incrementally process the new data + get the resulting changes.
        m_list = built_tn.process({click_source_str: click_m_list, customer_source_str: customer_m_list})

        # 3. Print out the size of the pydbsp state.
        sys.stdout.write(f"\rStep: {i + 1}, Memory: {built_tn.get_state_size() / 1024}KB")

        # 4. Add the changes to the output list.
        sink_m_list += m_list

    print()
    print(len(sink_m_list))
    print(sink_m_list[-10:])


In [ ]:
run()

This is of course not sustainable. It's time to introduce the `expire()` operator.

<a id="expire-operator"></a>
### expire()

Classic `map()` operator, like e.g. in Kafka Streams.

```
expire(ts_fun, expiry_fun, project_fun=lambda x: x[0], **kwargs)
```
* `ts_fun: r -> ts` timestamp function - gets a timestamp from the input record; gets an input record and returns a timestamp (any type; typically a Python `int`)
* `expiry_fun: ts -> ts` expiry function - gets a timestamp and returns the corresponding expiry timestamp specifying when the record shall expire
* `project_fun: tuple(r, ts) -> r` projection function - gets a pair of a record and its expiry timestamp and returns another record. Default: `lambda r_ts_tuple: r_ts_tuple[0]`

Let's see it in action.


In [ ]:
click_source_str = "clicks"
customer_source_str = "customers"

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    # <------------------------------>
    .expire(ts_fun=lambda r: r["ts"],
            expiry_fun=lambda ts: ts + click_generator.ts_step_int * 1000)
    # <------------------------------>
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

joined_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
)

built_tn = Tn.build(joined_tn)


Here, we use the `expiry()` operator to:
1. Select the `ts` field from each record,
2. and then set the expiry to the selected timestamp plus `1000` times the timestamp step size of the click generator.

You can observe multiple typical properties of expiry in Kafi Streams:
* It is defined once at the top of the topology, before any stateful operator (`map` and `filter` are stateless).
* The stateful operators (`distinct` and `join_equi`) below in the topology do not need to know anything about the expiry.

Ok. Let's see this in action. Will be able to rein in the memory consumption?

In [ ]:
for _ in range(10):
    run()

It works! Constant, flat memory usage!

Why? Because we used `expire()` to time out the transactional data (=the clicks). After a short while, the memory consumption of the master data (=the customers) becomes constant as well because it is limited (the generator only generates up to 100 customers in the example).

---
<a id="windows"></a>
## Time windows

In the previous section, we learnt how we can keep Kafi Streams' memory usage at check. It was only remotely related to time windows in the classical stream processing sense: under the covers, the `expire()` operator works akin to a "sliding window" in classical stream processing.

This section is about "real" stream processing time windows.

You'll see that we devised a novel formulation of them inside DBSP that allows us to build all the time window types from classical stream processing, but which is also so flexible that you can easily build your own custom time windows.


### Time Windows = expire + group/aggregate + trigger

What is a time window really? You can think of time windows in stream processing as consisting of three steps:
* **expire**: Time windows have a start and an end. Events *expire* after the end of a time window.
* **group/aggregate**: The actual "time window" is a set of events *grouped* by time and some other key (e.g. a customer ID) and *aggregated* (e.g. the number of purchases of a customer).
* **trigger**: The grouped and aggregated events are emitted based on a *trigger* mechanism (typically, when the window ends).

Now this is very theoretical. Let's pick the simplest time window - the *tumbling window* and see how all this theory plays out in practice.

In [22]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)
from kafi.streams.topologynode import TopologyNode as Tn

click_source_str = "clicks"
customer_source_str = "customers"

# <------------------------------>

def ts_fun(r):
    return r["ts"]

tumbling_size_int = click_generator.ts_step_int * 1000

# <------------------------------>

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    
    # <------------------------------>
    
    .expire_tumbling(ts_fun=ts_fun,
                     size_int=tumbling_size_int)
    
    # <------------------------------>
    
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

joined_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]})
)

# <------------------------------>

group_by_agg_tn = (
    joined_tn
    .group_by_agg_tumbling(size_int=tumbling_size_int,
                           ts_fun=ts_fun,
                           key_fun=lambda r: r["customer_id"],
                           agg_fun=lambda agg_r, r: {"clicks": agg_r["clicks"] + 1,
                                                     "total_view_time": agg_r["total_view_time"] + r["view_time"]},
                           agg_initial_any={"clicks": 0, "total_view_time": 0},
                           project_fun=lambda key_any, agg_r: {"customer_id": key_any,
                                                               "clicks": agg_r["clicks"],
                                                               "total_view_time": agg_r["total_view_time"]})
)

trigger_tn = (
    group_by_agg_tn
    .trigger(joined_tn,
             ts_fun=ts_fun)
)

# <------------------------------>

built_tn = Tn.build(trigger_tn)


In [24]:
run()

Step: 100, Memory: 1329.724609375KB
1000
[{'customer_id': 84, 'clicks': 11, 'total_view_time': 687}, {'customer_id': 3, 'clicks': 8, 'total_view_time': 639}, {'customer_id': 31, 'clicks': 7, 'total_view_time': 531}, {'customer_id': 34, 'clicks': 8, 'total_view_time': 608}, {'customer_id': 55, 'clicks': 5, 'total_view_time': 357}, {'customer_id': 68, 'clicks': 9, 'total_view_time': 712}, {'customer_id': 87, 'clicks': 9, 'total_view_time': 669}, {'customer_id': 97, 'clicks': 5, 'total_view_time': 399}, {'customer_id': 99, 'clicks': 1, 'total_view_time': 71}, {'customer_id': 29, 'clicks': 5, 'total_view_time': 452}]


In [ ]:
import random, uuid

customers_int = 10
ts_step_int = 1

class OrderGenerator:
    def __init__(self):
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = ts_step_int

    def generate(self):
        order_id_str = str(uuid.uuid4())
        m = {
            "key": order_id_str,
            "value": {"order_id": order_id_str,
                      "customer_id": random.randint(0, customers_int - 1),
                      "price": random.randint(1, 10000) / 100,
                      "ts": self.ts_int},
        }
        #
        self.ts_int += self.ts_step_int
        #
        return m

#

order_source_str = "orders"
sink_str = "sink"

#

def push(built_tn, customer_id, price, ts, w=1):
    r = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    built_tn.push("orders", [(r, w)])
    #
    sink_str_r_w_tuple_list_dict = built_tn.latest()
    #
    if sink_str_r_w_tuple_list_dict == {}:
        return []
    else:
        return sink_str_r_w_tuple_list_dict[sink_str]

def assert_output(actual_r_w_tuple_list, expected_r_w_tuple_list):
    def sort_key(r_w_tuple):
        r, w = r_w_tuple
        return (r.get("window_end", 0), r.get("customer_id", 0), w)
    #
    actual_sorted_r_w_tuple_list = sorted(actual_r_w_tuple_list, key=sort_key)
    expected_sorted_r_w_tuple_list = sorted(expected_r_w_tuple_list, key=sort_key)
    #
    if actual_sorted_r_w_tuple_list != expected_sorted_r_w_tuple_list:
        raise ValueError(f"\nAssertion Failed!\nExpected: {expected_sorted_r_w_tuple_list}\nGot:      {actual_sorted_r_w_tuple_list}")



In [ ]:
import sys
sys.path.insert(1, "../../..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

#

size_int = ts_step_int * 100
allowed_lateness_int = size_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_tumbling(lambda r: r["ts"], size_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_tumbling(size_int,
                                     lambda r: r["ts"],
                                     lambda r: r["customer_id"],
                                     lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                                       "total_price": agg_r["total_price"] + r["price"]},
                                     {"orders": 0, "total_price": 0},
                                     lambda by, agg_r: {"customer_id": by,
                                                        "orders": agg_r["orders"],
                                                        "total_price": agg_r["total_price"]},
                                     lambda r_end_ts_int_tuple: {**r_end_ts_int_tuple[0], "window_end": r_end_ts_int_tuple[1]},
                                     trigger_positive_only=False)
#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)

In [ ]:
built_tn.reset()

print("=== Step 1: Two orders from customer 1 (price=100, ts=10) and (price=200, ts=50) arrive ===")
push(built_tn, customer_id=1, price=100, ts=10, w=1)
push(built_tn, customer_id=1, price=200, ts=50, w=1)
print("-> OK")

print("\n=== Step 2: An order from customer 2 (price=50, ts=105) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=50, ts=105, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, 1)
])
print("-> OK: Window [0, 100) triggered (orders=2, total=300).")

print("\n=== Step 3: The order from customer 1 at 50 is retracted ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=200, ts=50, w=-1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, -1),
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, 1)
])
print("-> OK: Correction for window [0, 100) triggered: (customer=1, orders=1, total=100).")

print("\n=== Step 4: The order from customer 1 at 10 is also retracted ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=100, ts=10, w=-1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, -1)
])
print("-> OK: Retraction for window [0, 100) triggered.")

print("\n=== Step 5: An order from customer 3 (price=400, ts=100) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=3, price=400, ts=100, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order put into window [100, 200), still correctly held back/not triggered.")

print("\n=== Step 6: Another order from customer 3 at 150 (price=200, ts=150) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=3, price=200, ts=150, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK. Order put into window [100, 200), still correctly held back/not triggered.")

print("\n=== Step 7: Another order from customer 1 (price=50, ts=210) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=50, ts=210, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 2, "total_price": 600, "window_end": 200}, 1)
])
print("-> OK: Window [100, 200) triggered (customer=2, orders=1, total=50), (customer=3, orders=2, total=600).")

print("\n=== Step 8: An order from customer 2 (price=60, ts=310) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=60, ts=310, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 50, "window_end": 300}, 1)
])
print("-> OK: Window [200, 300) triggered (customer=1, orders=1, total=50).")

print("\n=== Step 9: Order from customer 2 (price=40, ts=120) arrives late (but not too late) ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=40, ts=120, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 200}, -1),
    ({"customer_id": 2, "orders": 2, "total_price": 90, "window_end": 200}, 1)
])
print("-> OK: Correction for window [100, 200) triggered: (customer=2, orders=2, total=90).")

print("\n=== Step 10: Order from customer 1 (price=70, ts=510) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=70, ts=510, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 2, "total_price": 90, "window_end": 200}, -1),
    ({"customer_id": 3, "orders": 2, "total_price": 600, "window_end": 200}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 60, "window_end": 400}, 1)
])
print("-> OK: Retraction for window [100, 200) triggered. Window (300, 400) triggered: (customer=2, orders=1, total=60).")

print("\n=== Step 11: Order from customer 1 (price=70, ts=130) arrives too late ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=70, ts=130, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")

print("\n🎉 Done.")

In [ ]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_hopping"
#
size_int = ts_step_int * 100
hop_int = size_int // 2
allowed_lateness_int = size_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_hopping(lambda r: r["ts"], size_int, hop_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_hopping(size_int,
                                    hop_int,
                                    lambda r: r["ts"],
                                    lambda r: r["customer_id"],
                                    lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                                      "total_price": agg_r["total_price"] + r["price"]},
                                    {"orders": 0, "total_price": 0},
                                    lambda by, agg_r: {"customer_id": by,
                                                       "orders": agg_r["orders"],
                                                       "total_price": agg_r["total_price"]},
                                    lambda r_end_ts_int_tuple: {**r_end_ts_int_tuple[0], "window_end": r_end_ts_int_tuple[1]},
                                    trigger_positive_only=False)
                                        
#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)

In [ ]:
built_tn.reset()

print("=== Step 1: Two orders from customer 1 arrive (ts=10 falls into [0, 100); ts=60 falls into [0, 100) and [50, 150)) ===")
push(built_tn, customer_id=1, price=100, ts=10, w=1)
push(built_tn, customer_id=1, price=200, ts=60, w=1)
print("-> OK.")

print("\n=== Step 2: An order from customer 2 at ts=110 arrives ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=50, ts=110, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, 1)
])
print("-> OK: Window [0, 100) triggered: (customer=1, orders=2, total=300)")

print("\n=== Step 3: Retraction for the window (50, 150) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=200, ts=60, w=-1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, -1),
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, 1)
])
print("-> OK: Correction for window [0, 100) triggered: (customer=1, orders=1, total=100).")

print("\n=== Step 4: An order from customer 3 at arrives (price=99, ts=150) ===")
r_w_tuple_list = push(built_tn, customer_id=3, price=99, ts=150, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 150}, 1)
])
print("-> OK: Window [50, 150) triggered: (customer=2, orders=2, total=50).")

print("\n=== Step 5: An order from customer 2 arrives (price=99, ts=250) ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=90, ts=250, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 1, "total_price": 99, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 1, "total_price": 99, "window_end": 250}, 1)
])
print("-> OK: Windows [100, 200) and [150, 250) triggered.")

print("\n=== Step 6: An order for customer 3 (price=100, ts=170) arrives late (but not too late) ===")
r_w_tuple_list = push(built_tn, customer_id=3, price=100, ts=170, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 3, "orders": 1, "total_price": 99, "window_end": 200}, -1),
    ({"customer_id": 3, "orders": 2, "total_price": 199, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 1, "total_price": 99, "window_end": 250}, -1),
    ({"customer_id": 3, "orders": 2, "total_price": 199, "window_end": 250}, 1)
])
print("-> OK: Corrections for windows [100, 200) and [150, 250) triggered.")

print("\n=== Step 7: An order for customer 1 (price=10, ts=450) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=10, ts=450, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 90, "window_end": 300}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 90, "window_end": 350}, 1)
])
print("-> OK: Window [0, 100) correctly expired; windows [200, 300) and [250, 300) correctly triggered.")

print("\n=== Step 8: An order from customer 1 (price=999, ts=20) arrives too late ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=999, ts=20, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")

print("\n🎉 Done.")


In [ ]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_cumulative"
#
size_int = ts_step_int * 100
advance_int = size_int // 5
allowed_lateness_int = size_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_cumulative(lambda r: r["ts"], size_int, advance_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_cumulative(size_int,
                                       advance_int,
                                       lambda r: r["ts"],
                                       lambda r: r["customer_id"],
                                       lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                                         "total_price": agg_r["total_price"] + r["price"]},
                                       {"orders": 0, "total_price": 0},
                                       lambda by, agg_r: {"customer_id": by,
                                                          "orders": agg_r["orders"],
                                                          "total_price": agg_r["total_price"]},
                                                          lambda r_end_ts_int_tuple: {**r_end_ts_int_tuple[0], "window_end": r_end_ts_int_tuple[1]},
                                       trigger_positive_only=False)

#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)


In [ ]:
built_tn.reset()

print("=== Step 1: An order for customer 1 arrives (price=100, ts=10). Lands in [0, 20), [0, 40), [0, 60), [0, 80), [0, 100) ===")
# 
push(built_tn, customer_id=1, price=100, ts=10, w=1)
print("-> OK.")


print("=== Step 2: Another order for customer 1 arrives (price=200, ts=30). Lands in [0, 40), [0, 60), [0, 80), [0, 100) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=200, ts=30, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 20}, 1)
])
print("-> OK: Window [0, 20) triggered.")


print("\n=== Step 3: An order for customer 2 arrives (price=50, ts=75). Lands in [60, 80), [80, 100) ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=50, ts=75, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 40}, 1),
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 60}, 1)
])
print("-> OK: Windows [0, 40) and [0, 60) triggered.")


print("\n=== Step 4: Another order from customer 1 (price=50, ts=15) arrives late (but not too late) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=50, ts=15, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 20}, -1),
    ({"customer_id": 1, "orders": 2, "total_price": 150, "window_end": 20}, 1),
    
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 40}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 40}, 1),
    
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 60}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 60}, 1)
])
print("-> OK: Corrections for windows [0, 20), [0, 40) and [0, 60) triggered.")


print("\n=== Step 5: Another order from customer 1 (price=500, ts=105) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=500, ts=105, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 80}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 80}, 1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 100}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 100}, 1)
])
print("-> OK: Windows [0, 80) and [0, 100] triggered.")


print("\n=== Step 6: Yet another order from customer 1 (price=10, ts=410) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=10, ts=410, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 150, "window_end": 20}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 40}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 60}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 80}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 80}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 100}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 100}, -1),
    #    
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 120}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 140}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 160}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 180}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 200}, 1)
])
print("-> OK: All windows until 200) triggered.")


print("\n=== Step 7: Another order from customer 1 (price=999, ts=10) arrives too late ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=999, ts=10, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")


print("\n🎉 Done.")


In [ ]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
sink_str = "orders_sliding"
#
size_int = ts_step_int * 100
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_sliding(lambda r: r["ts"], size_int)
)
#
window_tn = order_tn.window_sliding(size_int,
                                    lambda r: r["ts"],
                                    lambda r: r["customer_id"],
                                    lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                                      "total_price": agg_r["total_price"] + r["price"]},
                                    {"orders": 0, "total_price": 0},
                                    lambda by, agg_r: {"customer_id": by,
                                                       "orders": agg_r["orders"],
                                                       "total_price": agg_r["total_price"]},
                                    lambda r_end_ts_int_tuple: {**r_end_ts_int_tuple[0], "window_end": r_end_ts_int_tuple[1]},
                                    trigger_positive_only=False)

#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)


In [ ]:
built_tn.reset()

print("=== Step 1: An order from customer 1 arrives (price=100, ts=10) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=100, ts=10, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 110}, 1)
])
print("-> OK. Window [10, 110) triggered correctly.")


print("\n=== Step 2: Another order from customer 1 arrives (price=200, ts=30) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=200, ts=30, w=1)
assert_output(r_w_tuple_list, [
    ({'customer_id': 1, 'orders': 1, 'total_price': 100, 'window_end': 110}, -1),
    ({'customer_id': 1, 'orders': 2, 'total_price': 300, 'window_end': 110}, 1)]
)
print("-> OK: Correction for window [10, 110) triggered correctly.")


print("\n=== Step 3: An order from customer 2 arrives (price=50, ts=75) ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=50, ts=75, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 175}, 1)
])
print("-> OK: Window [75, 175) triggered correctly.")


print("\n=== Step 4: Another order from customer 1 arrives late but still inside [10, 110) (price=50, ts=15) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=50, ts=15, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 110}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 110}, 1)
])
print("-> OK: Window [10, 110) retracted correctly; window [15, 11ß) triggered correctly.")


print("\n=== Step 5: Yet another order from customer 1 arrives (not inside [15, 115) (price=500, ts=200) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=500, ts=200, w=1)
assert_output(r_w_tuple_list, [
    ({'customer_id': 1, 'orders': 3, 'total_price': 350, 'window_end': 110}, -1),
    ({'customer_id': 2, 'orders': 1, 'total_price': 50, 'window_end': 175}, -1),
    ({'customer_id': 1, 'orders': 1, 'total_price': 500, 'window_end': 300}, 1)
])
print("-> OK: Old windows [15, 115) and [75, 175) retracted correctly; wew window [200, 300) triggered correctly.")


print("\n=== Step 6: And yet another order from customer 1 arrives too late ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=999, ts=5, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")

print("\n🎉 Done.")

In [ ]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_session"
#
ts_step_int = 1
gap_int = ts_step_int * 20
max_session_int = ts_step_int * 200
allowed_lateness_int = gap_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_session(lambda r: r["ts"], max_session_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_session(gap_int,
                                    lambda r: r["ts"], 
                                    lambda r: r["customer_id"], 
                                    lambda agg_r, r: {"orders": agg_r["orders"] + 1,           
                                                      "total_price": agg_r["total_price"] + r["price"]},
                                    {"orders": 0, "total_price": 0},
                                    lambda by, agg_r: {"customer_id": by,
                                                       "orders": agg_r["orders"],
                                                       "total_price": agg_r["total_price"]},
                                    lambda r_end_ts_int_tuple: {**r_end_ts_int_tuple[0], "window_end": r_end_ts_int_tuple[1]},trigger_positive_only=False)

#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)


In [ ]:
built_tn.reset()

print("=== Step 1: An order from customer 1 arrives (price=100, ts=10) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=100, ts=10, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Window [10, 30) not yet triggered.")


print("\n=== Step 2: Another order from customer 1 arrives (price=200, ts=25) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=200, ts=25, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Window [25, 45) also not yet triggered but merged with [10, 30) => [10, 45).")


print("\n=== Step 3: An order from customer 2 arrives (price=50, ts=75) ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=50, ts=75, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 45}, 1)
])
print("-> OK: Window [10, 45) triggered.")


print("\n=== Step 4: Another order from customer 1 arrives (price=50, ts=15) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=50, ts=15, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 45}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 45}, 1)
])
print("-> OK: Correction for window [10, 45) triggered.")


print("\n=== Step 5: Yet another order from customer 1 arrives (price=500, ts=200) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=500, ts=200, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 95}, 1)
])
print("-> OK: Window [75, 95) triggered.")


print("\n=== Step 6: And yet another order from customer 1 arrives late but not too late (price=999, ts=1) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=999, ts=1, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 45}, -1),
    ({"customer_id": 1, "orders": 4, "total_price": 1349, "window_end": 45}, 1)
])
print("-> OK: Window [10, 45) retracted; New window [1, 45) triggered.")


print("\n=== Step 7: Yet another order from customer 1 arrives (price=100, ts=300) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=100, ts=300, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 4, "total_price": 1349, "window_end": 45}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 95}, -1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 220}, 1)
])
print("-> OK: Windows [1, 45) and [75, 95) retracted; window [200, 220) triggered.")

print("\n=== Step 8: An order from from customer 2 arrives too late (price=200, ts=2) ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=200, ts=2, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")

print("\n🎉 Done.")

In [ ]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_session_threshold"
#
ts_step_int = 1
gap_int = ts_step_int * 20
max_session_int = ts_step_int * 200
allowed_lateness_int = gap_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_session(lambda r: r["ts"], max_session_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_session(gap_int,
                                    lambda r: r["ts"], 
                                    lambda r: r["customer_id"], 
                                    lambda agg_r, r: {"orders": agg_r["orders"] + 1,           
                                                      "total_price": agg_r["total_price"] + r["price"]},
                                    {"orders": 0, "total_price": 0},
                                    lambda by, agg_r: {"customer_id": by,
                                                       "orders": agg_r["orders"],
                                                       "total_price": agg_r["total_price"]},
                                    lambda r_end_ts_int_tuple: {**r_end_ts_int_tuple[0], "window_end": r_end_ts_int_tuple[1]},
                                    trigger_fun=lambda r_end_ts_int_tuple, latest_ts_int: latest_ts_int >= r_end_ts_int_tuple[1] or r_end_ts_int_tuple[0]["total_price"] > 200, 
                                    trigger_positive_only=False)

#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)


In [ ]:
built_tn.reset()

print("=== Step 1: An order from customer 1 arrives (price=100, ts=10) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=100, ts=10, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Window [10, 30) not yet triggered.")


print("\n=== Step 2: Another order from customer 1 arrives (price=200, ts=25) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=200, ts=25, w=1)
assert_output(r_w_tuple_list, [
    ({'customer_id': 1, 'orders': 2, 'total_price': 300, 'window_end': 45}, 1)
])
print("-> OK: Window [25, 45) - and already triggered since total_price >= 300.")


print("\n=== Step 3: An order from customer 2 arrives (price=50, ts=75) ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=50, ts=75, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Window [10, 45) triggered.")


print("\n=== Step 4: Another order from customer 1 arrives (price=50, ts=15) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=50, ts=15, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 45}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 45}, 1)
])
print("-> OK: Correction for window [10, 45) triggered.")


print("\n=== Step 5: Yet another order from customer 1 arrives (price=500, ts=200) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=500, ts=200, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 95}, 1),
    ({'customer_id': 1, 'orders': 1, 'total_price': 500, 'window_end': 220}, 1)
])
print("-> OK: Windows [75, 95) and [200, 220) triggered (the latter has total price >=300).")


print("\n=== Step 6: And yet another order from customer 1 arrives late but not too late (price=999, ts=1) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=999, ts=1, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 45}, -1),
    ({"customer_id": 1, "orders": 4, "total_price": 1349, "window_end": 45}, 1)
])
print("-> OK: Window [10, 45) retracted; New window [1, 45) triggered.")


print("\n=== Step 7: Yet another order from customer 1 arrives (price=100, ts=300) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=100, ts=300, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 4, "total_price": 1349, "window_end": 45}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 95}, -1),
])
print("-> OK: Windows [1, 45) and [75, 95) retracted; window [200, 220) triggered.")

print("\n=== Step 8: An order from from customer 2 arrives too late (price=200, ts=2) ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=200, ts=2, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")

print("\n🎉 Done.")